In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print("train:", train.shape)
print("test :", test.shape)
train.head()

train: (5634, 40)
test : (1409, 39)


,Churn,customerID,Contract,OnlineBackup,MonthlyCharges,InternetService,StreamingTV,DeviceProtection,MultipleLines,TotalCharges,...,service_count,tenure_band,monthly_charge_band,bill_per_month,bill_pressure,household_score,profile_code,row_variant,dup_sig,col_order_key
0,0,8033-ATFAS,Month-to-month,No,60.61,DSL,Yes,Yes,No,2494.92,...,3,long,high,60.04,1.50,2,long_Month-to-month_Mailed check,shadow,TFAS_shadow,6917
1,0,6833-JMZYP,Month-to-month,No,94.00,Fiber optic,Yes,Yes,No,1505.45,...,4,mid,very_high,100.36,5.88,0,mid_Month-to-month_Credit card (automatic),base,MZYP_base,2482
2,0,4537-CIBHB,One year,No internet service,20.25,No,No internet service,No internet service,No,172.35,...,0,short,lower_mid,19.15,2.02,2,short_One year_Mailed check,base,IBHB_base,4145
3,0,1752-OZXFY,One year,No,59.80,DSL,Yes,No,No,3561.15,...,3,very_long,high,59.35,0.98,1,very_long_One year_Mailed check,base,ZXFY_base,3214
4,0,7622-FWGEW,Two year,Yes,85.65,DSL,Yes,Yes,No,4824.45,...,7,very_long,very_high,86.15,1.50,1,very_long_Two year_Bank transfer (automatic),base,WGEW_base,5801


In [2]:
train.info()


<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 40 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Churn                5634 non-null   int64  
 1   customerID           5634 non-null   str    
 2   Contract             5634 non-null   str    
 3   OnlineBackup         5536 non-null   str    
 4   MonthlyCharges       5634 non-null   float64
 5   InternetService      5634 non-null   str    
 6   StreamingTV          5634 non-null   str    
 7   DeviceProtection     5536 non-null   str    
 8   MultipleLines        5634 non-null   str    
 9   TotalCharges         5634 non-null   float64
 10  Dependents           5634 non-null   str    
 11  gender               5634 non-null   str    
 12  PaperlessBilling     5634 non-null   str    
 13  PhoneService         5634 non-null   str    
 14  TechSupport          5536 non-null   str    
 15  Partner              5634 non-null   str    
 16 

In [3]:
faltantes = train.isnull().sum()
print(faltantes[faltantes > 0].sort_values(ascending=False))

OnlineBackup        98
DeviceProtection    98
TechSupport         98
OnlineSecurity      98
PaymentMethod       73
support_depth       73
dtype: int64


In [4]:
print(list(train.columns))


['Churn', 'customerID', 'Contract', 'OnlineBackup', 'MonthlyCharges', 'InternetService', 'StreamingTV', 'DeviceProtection', 'MultipleLines', 'TotalCharges', 'Dependents', 'gender', 'PaperlessBilling', 'PhoneService', 'TechSupport', 'Partner', 'tenure', 'PaymentMethod', 'SeniorCitizen', 'StreamingMovies', 'OnlineSecurity', 'SeniorCitizenLabel', 'PartnerLabel', 'DependentsLabel', 'PhoneServiceLabel', 'PaperlessLabel', 'ServiceFamily', 'PaymentFamily', 'support_depth', 'stream_depth', 'service_count', 'tenure_band', 'monthly_charge_band', 'bill_per_month', 'bill_pressure', 'household_score', 'profile_code', 'row_variant', 'dup_sig', 'col_order_key']


In [5]:
# ¿los 98 faltantes de los servicios caen en las mismas filas?
cols_servicios = ['OnlineBackup', 'DeviceProtection', 'TechSupport', 'OnlineSecurity']
print(train[cols_servicios].isnull().all(axis=1).sum(), "filas con los 4 servicios faltantes a la vez")

# ¿PaymentMethod y support_depth faltan juntos?
print(train[['PaymentMethod', 'support_depth']].isnull().all(axis=1).sum(), "filas con ambos faltantes a la vez")

98 filas con los 4 servicios faltantes a la vez
73 filas con ambos faltantes a la vez


In [6]:
# para evaluar las columnas sospechosas
for col in ['profile_code', 'row_variant', 'dup_sig', 'col_order_key']:
    print(f"\n{col}: {train[col].nunique()} valores únicos")
    print(train[col].head(3).tolist())


profile_code: 58 valores únicos
['long_Month-to-month_Mailed check', 'mid_Month-to-month_Credit card (automatic)', 'short_One year_Mailed check']

row_variant: 2 valores únicos
['shadow', 'base', 'base']

dup_sig: 5599 valores únicos
['TFAS_shadow', 'MZYP_base', 'IBHB_base']

col_order_key: 5634 valores únicos
[6917, 2482, 4145]


In [7]:
# ¿cuántas base vs shadow?
print(train['row_variant'].value_counts())

# ¿las shadow son copias de las base? comparemos un cliente
# extraemos el "código base" del customerID (últimos 4 chars antes del variant)
print("\n--- ejemplo: buscamos si hay pares base/shadow ---")
print(train[['customerID', 'row_variant', 'Churn', 'tenure', 'MonthlyCharges']].head(10))

row_variant
base      5334
shadow     300
Name: count, dtype: int64

--- ejemplo: buscamos si hay pares base/shadow ---
   customerID row_variant  Churn  tenure  MonthlyCharges
0  8033-ATFAS      shadow      0      40           60.61
1  6833-JMZYP        base      0      15           94.00
2  4537-CIBHB        base      0       9           20.25
3  1752-OZXFY        base      0      60           59.80
4  7622-FWGEW        base      0      56           85.65
5  1078-TDCRN        base      0       3           30.75
6  4967-WPNCF        base      0      71           60.17
7  4559-UWIHT        base      0      14           82.65
8  6614-YWYSC        base      0      61           25.00
9  4664-NJCMS        base      0      33           59.55


In [8]:
# comparamos las shadow contra las base buscando duplicados en las columnas de datos
# (ignorando ids)
cols_id = ['customerID', 'dup_sig', 'col_order_key', 'row_variant', 'profile_code']
cols_datos = [c for c in train.columns if c not in cols_id]

# ¿cuántas filas están duplicadas si miramos solo los datos (sin ids)?
dups = train.duplicated(subset=cols_datos, keep=False).sum()
print(f"{dups} filas tienen datos idénticos a otra fila (ignorando IDs)")

# a ver si las shadow tienden a ser las duplicadas
print("\nDuplicados por tipo de fila:")
train['es_dup'] = train.duplicated(subset=cols_datos, keep=False)
print(train.groupby('row_variant')['es_dup'].sum())

24 filas tienen datos idénticos a otra fila (ignorando IDs)

Duplicados por tipo de fila:
row_variant
base      24
shadow     0
Name: es_dup, dtype: int64


In [9]:
# 1. ¿las shadow tienen una tasa de churn distinta a las base?
#    (si son muy distintas, podrían ser ruido que confunde)
print("Tasa de churn por tipo de fila:")
print(train.groupby('row_variant')['Churn'].mean())

# 2. ¿el test también tiene filas shadow? 
print("\n¿row_variant existe en test?", 'row_variant' in test.columns)
if 'row_variant' in test.columns:
    print(test['row_variant'].value_counts())

Tasa de churn por tipo de fila:
row_variant
base      0.267904
shadow    0.220000
Name: Churn, dtype: float64

¿row_variant existe en test? True
row_variant
base      1331
shadow      78
Name: count, dtype: int64


## Análisis de calidad de datos

### Valores faltantes
Se detectaron faltantes en 6 columnas con un patrón **sistemático** (no aleatorio):
- OnlineBackup, DeviceProtection, TechSupport, OnlineSecurity: 98 faltantes,
  ausentes en las **mismas 98 filas** (los 4 servicios faltan juntos).
- PaymentMethod y support_depth: 73 faltantes, también en las mismas filas.

El patrón indica un mecanismo común de ausencia, no aleatorio. [Tratamiento: pendiente]

### Auditoría de variables sin valor predictivo
- `col_order_key`: identificador único por fila (5634 valores únicos) → **se descarta**
  (un árbol podría memorizarlo y sobreajustar).
- `dup_sig`: identificador derivado (customerID + row_variant), 5599 únicos → **se descarta**.
- `profile_code`: concatenación de tenure_band + Contract + PaymentMethod (58 únicos),
  redundante con columnas existentes → **se descarta**.

### Filas "shadow" (row_variant)
El dataset contiene 300 filas marcadas como `shadow` (5.3%) además de 5334 `base`.
Investigación:
- **No son duplicados**: solo 24 filas tienen datos idénticos a otra, y todas son `base`.
  Las shadow tienen datos propios.
- **Tasa de churn similar**: base 26.8% vs shadow 22.0% → comportamiento comparable.
- **El test también contiene shadow** en proporción equivalente (78/1409 = 5.5%).

**Decisión**: se conservan las filas shadow, dado que el conjunto de test las incluye
en la misma proporción. Entrenar sin ellas degradaría la predicción sobre ese subconjunto
del test. La columna `row_variant` no se usará como variable predictiva (es una marca de
procedencia, no una característica del cliente).

In [10]:
# 1. ¿qué valores tienen las columnas de servicios? (para entender el "Unknown")
print(train['OnlineBackup'].value_counts(dropna=False))

# 2. ¿el test también tiene faltantes en estas columnas?
print("\nFaltantes en TEST:")
faltantes_test = test.isnull().sum()
print(faltantes_test[faltantes_test > 0].sort_values(ascending=False))

OnlineBackup
No                     2454
Yes                    1886
No internet service    1196
NaN                      98
Name: count, dtype: int64

Faltantes en TEST:
OnlineBackup        25
DeviceProtection    25
TechSupport         25
PaymentMethod       25
OnlineSecurity      25
support_depth       25
dtype: int64


In [11]:
# columnas categóricas con faltantes -> imputar con "Unknown"
cols_cat_faltantes = ['OnlineBackup', 'DeviceProtection', 'TechSupport', 
                      'OnlineSecurity', 'PaymentMethod']

for col in cols_cat_faltantes:
    train[col] = train[col].fillna("Unknown")
    test[col]  = test[col].fillna("Unknown")

# verificamos que ya no queden faltantes en esas columnas
print("Faltantes restantes en train:")
print(train[cols_cat_faltantes].isnull().sum())
print("\nFaltantes restantes en test:")
print(test[cols_cat_faltantes].isnull().sum())

Faltantes restantes en train:
OnlineBackup        0
DeviceProtection    0
TechSupport         0
OnlineSecurity      0
PaymentMethod       0
dtype: int64

Faltantes restantes en test:
OnlineBackup        0
DeviceProtection    0
TechSupport         0
OnlineSecurity      0
PaymentMethod       0
dtype: int64


In [12]:
# ¿qué pinta tiene support_depth?
print("Valores de support_depth:")
print(train['support_depth'].value_counts(dropna=False).sort_index())

print("\nTipo de dato:", train['support_depth'].dtype)

# ¿se relaciona con las columnas de soporte? veamos algunas filas
print("\nRelación con servicios de soporte:")
print(train[['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
            'TechSupport', 'support_depth']].head(8))

Valores de support_depth:
support_depth
0.0    2200
1.0    1184
2.0    1071
3.0     755
4.0     351
NaN      73
Name: count, dtype: int64

Tipo de dato: float64

Relación con servicios de soporte:
        OnlineSecurity         OnlineBackup     DeviceProtection  \
0                   No                   No                  Yes   
1                   No                   No                  Yes   
2  No internet service  No internet service  No internet service   
3                  Yes                   No                   No   
4                  Yes                  Yes                  Yes   
5                   No                  Yes                   No   
6                  Yes                  Yes                  Yes   
7                   No                  Yes                  Yes   

           TechSupport  support_depth  
0                   No            1.0  
1                   No            1.0  
2  No internet service            0.0  
3                   No        

In [13]:
servicios_soporte = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']

# recalculamos: contamos cuántos están en "Yes" (igual en train y test)
train['support_depth'] = (train[servicios_soporte] == "Yes").sum(axis=1)
test['support_depth']  = (test[servicios_soporte] == "Yes").sum(axis=1)

# verificamos que no queden faltantes y que coincida con los valores originales
print("Faltantes en support_depth → train:", train['support_depth'].isnull().sum(),
      "| test:", test['support_depth'].isnull().sum())
print("\nDistribución recalculada (train):")
print(train['support_depth'].value_counts().sort_index())

Faltantes en support_depth → train: 0 | test: 0

Distribución recalculada (train):
support_depth
0    2292
1    1191
2    1060
3     740
4     351
Name: count, dtype: int64


### Tratamiento de valores faltantes

El conjunto de **test presenta los mismos faltantes** que el train (25 por columna,
en las mismas 6 columnas), por lo que la estrategia debe funcionar en ambos conjuntos.
Esto **descarta la eliminación de filas**: no es posible borrar registros del test, ya
que se requiere predecir los 1409 clientes.

**Variables categóricas** (OnlineBackup, DeviceProtection, TechSupport, OnlineSecurity,
PaymentMethod): se imputó el faltante como una **categoría propia "Unknown"**, en lugar
de imputar con la moda. Justificación:
- El patrón de ausencia es sistemático (no aleatorio), por lo que asumir que esas filas
  "son como la mayoría" (moda) sería incorrecto.
- No se inventan datos: el modelo puede aprender si la ausencia misma se asocia al churn.
- Es coherente con el dataset, que ya trata estados especiales como categoría explícita
  (p. ej. "No internet service").

**Variable derivada `support_depth`**: se confirmó que es el **conteo de servicios de
soporte contratados** (OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport en "Yes").
En lugar de imputar sus 73 faltantes, se **recalculó** desde las columnas originales tras
la imputación, garantizando consistencia y eliminando los faltantes de forma controlada.

**Resultado**: el dataset queda sin valores faltantes en train ni test.